In [0]:
# COMMAND ----------
# 01 - XGBOOST FRAUD CLASSIFIER

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier
from skopt import BayesSearchCV
from skopt.space import Real, Integer
import mlflow
import mlflow.sklearn

df = spark.table("fraud.engineered_base").toPandas()

features = [
    "amount","avg_amount","invoice_count",
    "duplicate_flag","is_weekend","bank_change_flag",
    "ghost_vendor_flag","velocity_spike",
    "benford_score","high_amount_flag",
    "day_of_week","month"
]

X = df[features]
y = df["fraud_label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

search_space = {
    'max_depth': Integer(3, 8),
    'learning_rate': Real(0.01, 0.2, prior='log-uniform'),
    'subsample': Real(0.7, 1.0),
    'colsample_bytree': Real(0.7, 1.0),
    'n_estimators': Integer(100, 300),
    'reg_alpha': Real(0.0, 5.0),
    'reg_lambda': Real(0.0, 5.0)
}

xgb = XGBClassifier(eval_metric="auc", random_state=42)

opt = BayesSearchCV(
    estimator=xgb,
    search_spaces=search_space,
    n_iter=8,
    scoring='roc_auc',
    cv=2,
    random_state=42,
    verbose=1
)

with mlflow.start_run(run_name="xgb_fraud_bayessearch"):
    opt.fit(X_train, y_train)
    model = opt.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    auc = roc_auc_score(y_test, y_prob)
    print("\nBest Parameters:", opt.best_params_)
    print("\nXGBoost AUC:", auc)
    print(classification_report(y_test, y_pred))

    mlflow.log_metric("test_auc", auc)
    mlflow.xgboost.log_model(model, "xgb_model")

# Attach scores back to full df
df["fraud_probability"] = opt.best_estimator_.predict_proba(X)[:, 1]
spark.createDataFrame(df).write.mode("overwrite").saveAsTable("fraud.scored_xgb")
